In [29]:
from openai import OpenAI
import json
from dotenv import load_dotenv
from pprint import pprint

_ = load_dotenv()

client = OpenAI()

# 1. Define a list of callable tools for the model
tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "Get current weather for a specific city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The name of the city to get weather for",
                },
            },
            "required": ["city"],
        },
    },
]

def get_weather(city):
    weather_data = {
        "Paris": "18°C, ensoleillé avec quelques nuages",
        "Berlin": "12°C, pluvieux et nuageux",
        "Lille": "15°C, nuageux avec quelques éclaircies"
    }
    return weather_data.get(city, f"{city}: Données météo non disponibles")



In [30]:
#Create a running input list we will add to over time
input_list = [
    {"role": "user", "content": "Quel est le temps à Lille aujourd'hui?"}
]

# 2. Prompt the model with tools defined
response = client.responses.create(
    model="gpt-5",
    tools=tools,
    input=input_list,
)


pprint(response.output)
##Save function call outputs for subsequent requests
input_list += response.output


[ResponseReasoningItem(id='rs_68c28fbc076481908724befb8f1e30ab0115077a828abf6a', summary=[], type='reasoning', content=None, encrypted_content=None, status=None),
 ResponseFunctionToolCall(arguments='{"city":"Lille"}', call_id='call_3GYKcfkvbkcm5vFj80uZYZ6E', name='get_weather', type='function_call', id='fc_68c28fbd4f048190bd8d744a9d3ebd8c0115077a828abf6a', status='completed')]


In [31]:
for item in response.output:
    if item.type == "function_call":
        if item.name == "get_weather":
            # 3. Execute the function logic for get_horoscope
            weather = get_weather(**json.loads(item.arguments))
            
            # 4. Provide function call results to the model
            input_list.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps({
                  "get_weather": weather
                })
            })

pprint("Final input:")
pprint(input_list)

'Final input:'
[{'content': "Quel est le temps à Lille aujourd'hui?", 'role': 'user'},
 ResponseReasoningItem(id='rs_68c28fbc076481908724befb8f1e30ab0115077a828abf6a', summary=[], type='reasoning', content=None, encrypted_content=None, status=None),
 ResponseFunctionToolCall(arguments='{"city":"Lille"}', call_id='call_3GYKcfkvbkcm5vFj80uZYZ6E', name='get_weather', type='function_call', id='fc_68c28fbd4f048190bd8d744a9d3ebd8c0115077a828abf6a', status='completed'),
 {'call_id': 'call_3GYKcfkvbkcm5vFj80uZYZ6E',
  'output': '{"get_weather": "15\\u00b0C, nuageux avec quelques '
            '\\u00e9claircies"}',
  'type': 'function_call_output'}]


In [33]:
response = client.responses.create(
    model="gpt-5",
    # instructions="Respond only with the weather generated by a tool.",
    tools=tools,
    input=input_list,
)

# 5. The model should be able to give a response!
print("Final output:")
print(response.model_dump_json(indent=2))
print("\n" + response.output_text)


Final output:
{
  "id": "resp_68c2900209448190b27f3fc252250cdf0115077a828abf6a",
  "created_at": 1757581314.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5-2025-08-07",
  "object": "response",
  "output": [
    {
      "id": "msg_68c290028b608190886e39e6075dea0e0115077a828abf6a",
      "content": [
        {
          "annotations": [],
          "text": "À Lille aujourd’hui : 15°C, nuageux avec quelques éclaircies.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [
    {
      "name": "get_weather",
      "parameters": {
        "type": "object",
        "properties": {
          "city": {
            "type": "string",
            "description": "The name of the city to get weather for"
          }
        },
        

In [ ]:
from pydantic_ai import Agent

agent = Agent(
    'gpt-5'
)

@agent.tool_plain  
def get_weather(city) -> str:
    """Get current weather for a specific city."""
    weather_data = {
        "Paris": "18°C, ensoleillé avec quelques nuages",
        "Berlin": "12°C, pluvieux et nuageux",
        "Lille": "15°C, nuageux avec quelques éclaircies"
    }
    return weather_data.get(city, f"{city}: Données météo non disponibles")

result = agent.run_sync('Quel est le temps à Lille aujourd\'hui?') 

print(result)

RuntimeError: This event loop is already running